# Lab 09 Real: OOP and Einsum

**Course:** MATH 170 -- Introduction to Scientific Computing  
**Term:** Spring 2026

In this lab, you will complete two short tasks from Week 12: basic OOP and a batched tensor computation with `torch.einsum()`.

**Instructions:**
1. Run the setup cells in order.
2. Edit the `submission.py` cell or edit the file locally.
3. Run the preview cells to check your work.
4. Run the `pytest` cell to verify your solution.


## Setup Instructions

### If using Google Colab:
1. Run each cell from top to bottom.
2. The notebook will create `submission.py` and `test_submission.py`.
3. Edit the `submission.py` cell with your solution.
4. Run the preview and test cells.
5. Download `submission.py` and submit via GitHub.

### If working locally:
1. You can edit `submission.py` directly in your editor.
2. Run the preview cells in the notebook if you want to check your work.
3. Run `pytest test_submission.py -v` in this folder to test.


In [1]:
import importlib.util
import subprocess
import sys

for pkg in ['torch', 'pytest']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)


In [2]:
%%writefile test_submission.py
"""
Test cases for Lab 09 Real - DO NOT MODIFY
"""

import torch
from submission import Vector2D, batch_right_multiply_einsum


def test_vector2d_stores_attributes():
    v = Vector2D(3, -1)
    assert v.x == 3
    assert v.y == -1


def test_vector2d_str():
    v = Vector2D(2, 5)
    assert str(v) == "Vector2D(2, 5)"


def test_vector2d_add_returns_new_vector():
    v1 = Vector2D(1, 2)
    v2 = Vector2D(-3, 4)
    result = v1.add(v2)
    assert isinstance(result, Vector2D)
    assert result.x == -2
    assert result.y == 6


def test_vector2d_dot():
    v1 = Vector2D(1, 2)
    v2 = Vector2D(3, 4)
    assert v1.dot(v2) == 11


def test_batch_right_multiply_einsum_shape():
    T = torch.arange(24, dtype=torch.float32).reshape(2, 3, 4)
    B = torch.arange(20, dtype=torch.float32).reshape(4, 5)
    result = batch_right_multiply_einsum(T, B)
    assert result.shape == (2, 3, 5)


def test_batch_right_multiply_einsum_matches_slice_by_slice_matmul():
    T = torch.tensor(
        [
            [[1.0, 2.0], [3.0, 4.0]],
            [[0.0, 1.0], [1.0, 0.0]],
        ]
    )
    B = torch.tensor(
        [
            [10.0, 20.0, 30.0],
            [1.0, 2.0, 3.0],
        ]
    )
    result = batch_right_multiply_einsum(T, B)
    expected = torch.stack([T[0] @ B, T[1] @ B], dim=0)
    assert torch.equal(result, expected)


def test_batch_right_multiply_einsum_batch_values():
    T = torch.tensor(
        [
            [[1.0, 0.0, 2.0]],
            [[0.0, 1.0, 1.0]],
        ]
    )
    B = torch.tensor(
        [
            [1.0, 2.0],
            [3.0, 4.0],
            [5.0, 6.0],
        ]
    )
    result = batch_right_multiply_einsum(T, B)
    expected = torch.tensor(
        [
            [[11.0, 14.0]],
            [[8.0, 10.0]],
        ]
    )
    assert torch.equal(result, expected)


Writing test_submission.py


---
## Task 1: Basic OOP

Week 12A built a `Vector` class with attributes and methods. Here we keep only the most basic pieces: store coordinates, print a readable string, add two vectors, and compute a dot product.


In [3]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f'Dog({self.name}, {self.age})'

buddy = Dog('Buddy', 3)
print(buddy)
print(buddy.name)


Dog(Buddy, 3)
Buddy


---
## Task 2: Batched `einsum`

Let `T` be a tensor with shape `(b, m, n)`. Think of it as a batch of matrices

`[A_0, A_1, ..., A_{b-1}]`

where `T[i, :, :] = A_i` has shape `(m, n)`. Let `B` be a matrix with shape `(n, k)`.

Your goal is to return a tensor `C` with shape `(b, m, k)` such that

`C[i, :, :] = A_i @ B`

for every batch index `i`. Use `torch.einsum()`.


In [4]:
import torch

T = torch.tensor([
    [[1.0, 2.0], [3.0, 4.0]],
    [[0.0, 1.0], [1.0, 0.0]],
])
B = torch.tensor([
    [10.0, 20.0, 30.0],
    [1.0, 2.0, 3.0],
])

print(torch.einsum('bmn,nk->bmk', T, B))


tensor([[[ 12.,  24.,  36.],
         [ 34.,  68., 102.]],

        [[  1.,   2.,   3.],
         [ 10.,  20.,  30.]]])


---
## Tasks

Implement the following in `submission.py`:

1. `Vector2D`
   - store `x` and `y`
   - define `__str__`
   - define `add(other)`
   - define `dot(other)`

2. `batch_right_multiply_einsum(T, B)`
   - `T` has shape `(b, m, n)`
   - `B` has shape `(n, k)`
   - return a tensor with shape `(b, m, k)`
   - use `torch.einsum`


In [5]:
%%writefile submission.py
"""
MATH 170 - Lab 09 Real: OOP and Einsum
Spring 2026

Instructions:
1. Implement a small Vector2D class using basic OOP.
2. Use torch.einsum for a batched tensor-matrix multiplication.
"""

import torch


class Vector2D:
    """
    Simple 2D vector class.

    Attributes
    ----------
    x : float
        First coordinate.
    y : float
        Second coordinate.
    """

    def __init__(self, x, y):
        """Store the coordinates on the instance."""
        # TODO: store x and y as instance attributes
        self.x = x
        self.y = y

    def __str__(self):
        """Return a readable string like Vector2D(3, 4)."""
        # TODO: return the string representation
        return f"Vector2D({self.x}, {self.y})"

    def add(self, other):
        """
        Return a new Vector2D equal to self + other.

        Parameters
        ----------
        other : Vector2D
            Another vector.
        """
        # TODO: return a new Vector2D with added coordinates
        return Vector2D(self.x + other.x, self.y + other.y)

    def dot(self, other):
        """
        Return the dot product self.x * other.x + self.y * other.y.

        Parameters
        ----------
        other : Vector2D
            Another vector.
        """
        # TODO: return the dot product
        return self.x * other.x + self.y * other.y


def batch_right_multiply_einsum(T: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """
    Return the batched matrix product C using torch.einsum.

    Think of T as a batch of matrices:

        [A_0, A_1, ..., A_{b-1}]

    If T has shape (b, m, n), then each A_i = T[i, :, :] has shape (m, n).
    Let B have shape (n, k).

    Return a tensor C with shape (b, m, k) such that

        C[i, :, :] = A_i @ B

    for every batch index i.

    Use torch.einsum.
    """
    # TODO: use torch.einsum for batched right multiplication
    return torch.einsum('bmn,nk->bmk', T, B)


Writing submission.py


In [6]:
import importlib
import submission
importlib.reload(submission)

from submission import Vector2D, batch_right_multiply_einsum


---
## Preview Your Work

These cells are for debugging and quick checks.


In [7]:
try:
    v1 = Vector2D(1, 2)
    v2 = Vector2D(3, 4)
    print(v1)
    print(v1.add(v2))
    print(v1.dot(v2))
except Exception as e:
    print('Vector2D not implemented yet:', e)


Vector2D(1, 2)
Vector2D(4, 6)
11


In [8]:
T = torch.tensor([
    [[1.0, 2.0], [3.0, 4.0]],
    [[0.0, 1.0], [1.0, 0.0]],
])
B = torch.tensor([
    [10.0, 20.0, 30.0],
    [1.0, 2.0, 3.0],
])

try:
    print(batch_right_multiply_einsum(T, B))
except Exception as e:
    print('batch_right_multiply_einsum not implemented yet:', e)


tensor([[[ 12.,  24.,  36.],
         [ 34.,  68., 102.]],

        [[  1.,   2.,   3.],
         [ 10.,  20.,  30.]]])


---
## Run the Public Tests

This cell runs the same public tests contained in `test_submission.py`.


In [9]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('pytest') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)

subprocess.run([sys.executable, '-m', 'pytest', 'test_submission.py', '-v'], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', 'test_submission.py', '-v'], returncode=0)

---
## Submission

### Option 1: Google Colab
1. Download `submission.py` from the Colab file panel.
2. Upload it to your GitHub lab repository.
3. Commit and push.

### Option 2: Local
```bash
git add submission.py
git commit -m "Complete Lab 09 Real"
git push
```

Check GitHub Actions for the final autograder result.
